In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import FancyBboxPatch
import gzip
from scipy.optimize import curve_fit
import os
import matplotlib.image as mpimg
from matplotlib.patches import FancyBboxPatch

In [2]:
###########################################################

file_dir = os.getcwd()

os.chdir(file_dir)

###########################################################

In [3]:
# PARAMETER DEFINITIONS AND LOADING DATA FOR PLOTTING

df = pd.read_csv('../QSim_Data/09-1265A-E_Advantage_system5_4_annealing_schedule.csv', sep=',', decimal='.')
sa = np.array([round(df['s'][i],3) for i in range(1000)])
Ga = np.array([round(df['A(s) (GHz)'][i],3)*np.pi for i in range(1000)])
Ja = np.array([round(df['B(s) (GHz)'][i],3)*np.pi for i in range(1000)])

def sc(E,gc=1.):
    return sa[np.argmin(abs(Ga-gc*E*Ja))]
def Jc(E,gc=1.):
    return E * Ja[np.argmin(abs(Ga-gc*E*Ja))]
def Gc(E,gc=1.):
    return Ga[np.argmin(abs(Ga-gc*E*Ja))]
def Jcp(E,gc=1.):
    return np.gradient(E*Ja,sa)[np.argmin(abs(Ga-gc*E*Ja))]
def Gcp(E,gc=1.):
    return np.gradient(Ga,sa)[np.argmin(abs(Ga-gc*E*Ja))]
def tQ(ta,E,gc=1.):
    # ta must be in ns
    return ta / gc * Jc(E,gc) / (Jcp(E,gc)/Jc(E,gc) - Gcp(E,gc)/Gc(E,gc))
def tQ_to_ta(tq,E,gc=1.):
    # ta must be in ns
    return tq * gc / Jc(E,gc) * (Jcp(E,gc)/Jc(E,gc) - Gcp(E,gc)/Gc(E,gc))
def hcoJc(h,E,gc=1.):
    return h
def hcoJc_to_hz(hcoJc,E,gc=1.):
    return hcoJc
def en_to_lambda(E,gc=1.):
    return ((Jcp(E,gc)/Jc(E,gc) - Gcp(E,gc)/Gc(E,gc)) / Jc(E,gc)) / ((Jcp(1.,gc)/Jc(1.,gc) - Gcp(1.,gc)/Gc(1.,gc)) / Jc(1.,gc))

en_range = np.linspace(.1,2.,20).round(6)
            
ens, ens1, tqs, tas, hcojs, hzs = [], [], [], [], [], []
for en in en_range:
    for tq in np.geomspace(tQ(5,.1),tQ(100,2.),40):
        if tQ(5,en) < tq < tQ(100,en):
            for hz in np.concatenate((np.geomspace(hcoJc(-.1,.1), hcoJc(-.01,.1), 12),
                                      np.geomspace(hcoJc(-1.,.1), hcoJc(-.11,.1), 12))):
                if hcoJc(-1.,en) <= hz <= hcoJc(-0.,en):
                    hcojs.append(hz.round(6))
                    hzs.append(hcoJc_to_hz(hz,en).round(6))
                    ens.append(en.round(6))
                    tqs.append(tq.round(6))
                    tas.append((tQ_to_ta(tq,en)/1000.).round(6))

L = 5564
J = -1.

hzenta_range = [[ens[i],tas[i],hzs[i]] for i in range(len(ens))]

profil = 'jvodebjuelich'
num_reads = 1000
embedding_string_short = '54chainPBC'
auto_scale = False
answer_mode="raw"
fast_anneal=True

#ds = range(0,L//2+1)
ds = range(0,1)

# READ ANALYZED RESULTS

with gzip.open('../QSim_Data/1D_tfim_1st_sweep_mag.txt.gz', "r") as f:
    fvd_mags = eval(f.read())
with gzip.open('../QSim_Data/1D_tfim_1st_sweep_corr.txt.gz', "r") as f:
    fvd_corrs = eval(f.read())

# PLOTTING CONVERSION

tqsu = np.unique(tqs)
hcojcsu = np.unique(hcojs)
datamag = {}
datanex = {}
keys = []
for tq in tqsu:
    for hcojc in hcojcsu:
        datamag[(tq,hcojc)] = []
        datanex[(tq,hcojc)] = []
        keys.append((tq,hcojc))

for hzentaind, [en,ta,hz] in enumerate(hzenta_range):

    try:
        tq = tqsu[np.abs(tQ(1000*ta,en).round(6) - tqsu).argmin()]
        hcojc = hcojcsu[np.abs(hcoJc(hz,en) - hcojcsu).argmin()]
        datamag[(tq,hcojc)].append([en_to_lambda(en),fvd_mags[(L,J,hz,ta,en,num_reads,'szmean')]])
        datanex[(tq,hcojc)].append([en_to_lambda(en),fvd_mags[(L,J,hz,ta,en,num_reads,'nexmean')]])
    except:
        continue

for key in keys:
    if datamag[key] == []:
        del datamag[key]
        del datanex[key]
        print(key)

# PLOTTING CONVERSION

mag = {}
nex = {}
corr = {}

for hzentaind, [en,ta,hz] in enumerate(hzenta_range):
    try:
        mag[(hz,ta,en)] = fvd_mags[(L,J,hz,ta,en,num_reads,'szmean')]
        nex[(hz,ta,en)] = fvd_mags[(L,J,hz,ta,en,num_reads,'nexmean')]

        for d in ds:
            corr[(hz,ta,en,d)] = fvd_corrs[(L,J,hz,ta,en,num_reads,d,'szsz'+str(d)+'mean')]
    except:
        continue

In [4]:
# Define scale factors and colormap
factors = np.linspace(0.1, 2.0, 10)
norm = mcolors.Normalize(vmin=min(factors), vmax=max(factors))
cmap = cm.viridis
colors = [cmap(norm(f)) for f in factors]

# Create figure and axis
ms = 7.5
fs = 20
lw = .5
s = 10

plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8.0, 6.0))
# fig, ax = plt.subplots(figsize=(4, 3))
ax.set_xlabel('$s$', fontsize=fs)
ax.set_ylabel('Energy~(GHz)', fontsize=fs)
ax.tick_params(axis='both', which='major', labelsize=fs)

# Plot A(s)
ax.plot(sa, Ga, label='$A(s)$', color='black', linestyle='--', lw=2)

# Plot scaled B(s)
for i, f in enumerate(factors):
    ax.plot(sa, f * Ja, color=colors[i], lw=2)

# Legend: only A(s) and B(s)
custom_lines = [
    plt.Line2D([0], [0], color='black', linestyle='--', lw=2, label='$A(s)$'),
    plt.Line2D([0], [0], color='gray', linestyle='-', lw=2, label='$B(s)$')
]
ax.legend(handles=custom_lines, fontsize=fs, loc='best', bbox_to_anchor=(0.5, 0.75), frameon=False)

# Colorbar for scaling of B(s)
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02, fraction=0.15, aspect=35)
cbar.set_label(label='Scaling factor $E$', size=fs)
cbar.ax.tick_params(labelsize=fs)

# --- Combined box, arrow and text for lambda/tauQ ---
arrow_x = 0.6
arrow_y = 2.75
arrow_dy = 10

# Dimensions for the box
box_left = arrow_x + .12
box_bottom = arrow_y - 1.
box_width = 0.01
box_height = arrow_dy + 1

# Draw background box manually
box = FancyBboxPatch(
    (box_left, box_bottom), box_width, box_height,
    boxstyle="round,pad=0.2", edgecolor='black', facecolor='white', alpha=0.7, zorder=2
)
ax.add_patch(box)

# Add arrow inside the box
ax.annotate(
    '', xy=(arrow_x, arrow_y + arrow_dy), xytext=(arrow_x, arrow_y),
    arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=3
)

# Add text inside the box
ax.text(
    arrow_x + 0.05, arrow_y + arrow_dy - 6.,
    r'$\lambda \to 0$' + '\n' + r'$\tau_Q \to \infty$',
    fontsize=fs+4, verticalalignment='center', zorder=3
)

# --- Horizontal arrow for t_a ---
ta_arrow_y = -2.5
ax.annotate(
    '', xy=(0.5, ta_arrow_y), xytext=(0.2, ta_arrow_y),
    arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=1
)

ax.text(
    .15, ta_arrow_y ,
    r'$t_a$', fontsize=fs+4, ha='center', va='center', zorder=3,
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='black', alpha=0.7)
)

# Final formatting
ax.set_ylim(-5., 30.)
fig.tight_layout()
fig.savefig('AnnealingScheduleTauLambda_paper.pdf')
plt.show()

/tmp/ipykernel_35522/1560058841.py:96: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# PLOT RESULTS

############################ MAIN FIGURE ############################

def fit_func(x, C0, C2):
    return C0 + C2*x**2

third_axis = tqsu[:34]
third_axis_name = r'$\tau_Q$'
# cmap = plt.get_cmap('gnuplot')
cmap = cm.viridis
colors = [cmap(i) for i in np.linspace(0., 1., len(third_axis))]

ms = 7.5
fs = 20
lw = .5
s = 10

znemag = np.full((len(hcojcsu),len(third_axis)),-2.)
znenex = np.full((len(hcojcsu),len(third_axis)),-2.)

#for hcojcind, hcojc in enumerate(hcojcsu):
hcojcind = 12
hcojc = hcojcsu[hcojcind]
    
plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12.0, 6.0))

ax[0].set_xlabel(r"$\lambda$", fontsize = fs)
ax[0].tick_params(axis='x',labelsize=fs)
ax[0].tick_params(axis='y',labelsize=fs)
ax[0].set_ylabel(r"$\langle \sigma^z \rangle$", fontsize=fs)
# ax[0].set_title('$L$ = '+str(L)+' chain PBC\n$h$ = '+str(hcojc), fontsize = 23)

ax[1].set_xlabel(r"$\lambda$", fontsize = fs)
ax[1].tick_params(axis='x',labelsize=fs)
ax[1].tick_params(axis='y',labelsize=fs)
ax[1].set_ylabel(r"$\langle n_\mathrm{ex}\rangle$", fontsize=fs)
# ax[1].set_title('$L$ = '+str(L)+' chain PBC\n$h$ = '+str(hcojc), fontsize = 23)

#####################################################

# --- ADD THIS TO CREATE AND POPULATE THE INSET ---
# 1. Create the inset axes in ax[0]
# You may need to tweak the first two numbers (x, y) to move the box 
# so it doesn't overlap with your main scatter plot data.
axins = ax[0].inset_axes([0.45, 0.55, 0.5, 0.4]) 

# 2. Plot your lambda vs E curve into the inset
# I added color='black' so it stands out, but you can change this
axins.plot(en_range, [en_to_lambda(e,gc=3.04438) for e in en_range], color='black')

# 3. Format the inset axes
axins.set_xlabel('$E$', fontsize=fs-4)
axins.set_ylabel(r'$\lambda$', fontsize=fs-4)
axins.tick_params(axis='x', labelsize=fs-4)
axins.tick_params(axis='y', labelsize=fs-4)

#####################################################

for tqi, tq in enumerate(third_axis):
    
    try:
        x = np.array(datamag[(tq,hcojc)])[:,0]
        y = np.array(datamag[(tq,hcojc)])[:,1]
        params = curve_fit(fit_func, x, y)
        [C0, C2] = params[0]
        ax[0].scatter(x, y, color=colors[tqi], s=s)
        x = np.linspace(0.,x[0],100)
        y = [min([fit_func(xi, C0, C2),1.]) for xi in x]
        ax[0].plot(x, y, color=colors[tqi], lw=lw)
        
        znemag[hcojcind,tqi] = min([fit_func(0., C0, C2),1.])

        x = np.array(datanex[(tq,hcojc)])[:,0]
        y = np.array(datanex[(tq,hcojc)])[:,1]
        params = curve_fit(fit_func, x, y)
        [C0, C2] = params[0]
        ax[1].scatter(x, y, color=colors[tqi], s=s)
        x = np.linspace(0.,x[0],100)
        y = [max([fit_func(xi, C0, C2),0.]) for xi in x]
        ax[1].plot(x, y, color=colors[tqi], lw=lw)
        
        znenex[hcojcind,tqi] = fit_func(0., C0, C2)
    except:
        znemag[hcojcind,tqi] = np.nan
        znenex[hcojcind,tqi] = np.nan

#fig.tight_layout(pad=2.5)
normalize = mcolors.Normalize(vmin=min(abs(third_axis)), vmax=max(abs(third_axis)))
scalarmappaple = cm.ScalarMappable(norm=normalize, cmap=cmap)
scalarmappaple.set_array(third_axis)
# cbar = plt.colorbar(
#     scalarmappaple, 
#     ax=ax.ravel().tolist(), 
#     pad=0.05,       # Pushes the colorbar to the right (increase this number to move it further)
#     fraction=0.025   # Controls the thickness of the colorbar itself
# )
# cbar.set_label(label=third_axis_name, size=fs)
# cbar.ax.tick_params(labelsize=fs)

cbar_ax = fig.add_axes([0.15, 0.125, 0.7, 0.02]) 
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(scalarmappaple, cax=cbar_ax, orientation='horizontal', pad=0.1, fraction=0.15, aspect=10)
cbar.set_label(third_axis_name, fontsize=fs, labelpad=10)
cbar.ax.tick_params(labelsize=fs)
#axsz.set_xscale('log')
#axsz.set_yscale('log')
#axnex.set_xscale('log')
#axnex.set_yscale('log')

fig.tight_layout(rect=[0, 0.1, 1, 1]) # 
fig.savefig('1D_ZNE_extrapolation_hz_'+str(hcojc)+'_paper.pdf')

/tmp/ipykernel_35522/1757210391.py:73: OptimizeWarning: Covariance of the parameters could not be estimated
  params = curve_fit(fit_func, x, y)
/tmp/ipykernel_35522/1757210391.py:84: OptimizeWarning: Covariance of the parameters could not be estimated
  params = curve_fit(fit_func, x, y)
/tmp/ipykernel_35522/1757210391.py:119: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0.1, 1, 1]) #


In [6]:
# ==========================================
# 1. GLOBAL SETTINGS & DATA LOADING
# ==========================================
fs = 24
ms = 7.5
lw = 0.5
s = 10

plt.switch_backend('pgf')
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": fs,
})

def fit_func(x, C0, C2): return C0 + C2*x**2

# ==========================================
# 2. CREATE MASTER FIGURE & GRID LAYOUT
# ==========================================
# Make the figure larger to accommodate the 2x2 grid
fig = plt.figure(figsize=(14.0, 12.0))

# Create a 2x2 grid. 
gs = fig.add_gridspec(nrows=2, ncols=2, hspace=0.25, wspace=0.25)

ax_tl = fig.add_subplot(gs[0, 0]) # Top Left
ax_tr = fig.add_subplot(gs[0, 1]) # Top Right
ax_bl = fig.add_subplot(gs[1, 0]) # Bottom Left
ax_br = fig.add_subplot(gs[1, 1]) # Bottom Right

# ==========================================
# 3. TOP-LEFT PLOT: Annealing Schedule
# ==========================================
factors1 = np.linspace(0.1, 2.0, 10)
norm1 = mcolors.Normalize(vmin=min(factors1), vmax=max(factors1))
cmap1 = cm.viridis
colors1 = [cmap1(norm1(f)) for f in factors1]

ax_tl.set_xlabel('$s$', fontsize=fs)
ax_tl.set_ylabel('Energy~(GHz)', fontsize=fs)
ax_tl.tick_params(axis='both', which='major', labelsize=fs)

ax_tl.plot(sa, Ga, label='$A(s)$', color='black', linestyle='--', lw=2)

for i, f in enumerate(factors1):
    ax_tl.plot(sa, f * Ja, color=colors1[i], lw=2)

custom_lines = [
    plt.Line2D([0], [0], color='black', linestyle='--', lw=2, label='$A(s)$'),
    plt.Line2D([0], [0], color='gray', linestyle='-', lw=2, label='$B(s)$')
]
ax_tl.legend(handles=custom_lines, fontsize=fs-2, loc='best', bbox_to_anchor=(0.585, 0.745), frameon=False)

sm1 = cm.ScalarMappable(cmap=cmap1, norm=norm1)
sm1.set_array([])
cbar1 = plt.colorbar(sm1, ax=ax_tl, pad=0.02, fraction=0.15, aspect=35)
cbar1.set_label(label='Scaling factor $E$', size=fs)
cbar1.ax.tick_params(labelsize=fs)

arrow_x, arrow_y, arrow_dy = 0.6, 2.75, 10
box = FancyBboxPatch(
    (arrow_x + .15, arrow_y - 1.), 0.01, arrow_dy + 1,
    boxstyle="round,pad=0.2", edgecolor='black', facecolor='white', alpha=0.7, zorder=2
)
ax_tl.add_patch(box)
ax_tl.annotate('', xy=(arrow_x, arrow_y + arrow_dy), xytext=(arrow_x, arrow_y), arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=3)
ax_tl.text(arrow_x + 0.05, arrow_y + arrow_dy - 6., r'$\lambda \to 0$' + '\n' + r'$\tau_Q \to \infty$', fontsize=fs-2, verticalalignment='center', zorder=3)

ta_arrow_y = -2.5
ax_tl.annotate('', xy=(0.5, ta_arrow_y), xytext=(0.2, ta_arrow_y), arrowprops=dict(arrowstyle='->', lw=2, color='black'), zorder=1)
ax_tl.text(.15, ta_arrow_y, r'$t_a$', fontsize=fs-2, ha='center', va='center', zorder=3, bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='black', alpha=0.7))
ax_tl.set_ylim(-5., 30.)

# ==========================================
# 4. TOP-RIGHT PLOT: "some_plot.pdf"
# ==========================================
# Note: You MUST convert the PDF to a PNG image to render it here.
try:
    img = mpimg.imread('1D_embedding.png') 
    ax_tr.imshow(img)
    ax_tr.axis('off') # Hides the axes/ticks around the image
except FileNotFoundError:
    ax_tr.text(0.5, 0.5, "Convert '1D_embedding.pdf'\nto '1D_embedding.png'\nto display here.", 
               ha='center', va='center', fontsize=fs)
    ax_tr.axis('off')

# ==========================================
# 5. BOTTOM PLOTS: 1D ZNE extrapolation
# ==========================================
third_axis = tqsu[:34]
third_axis_name = r'$\tau_Q$'
cmap2 = cm.viridis
colors2 = [cmap2(i) for i in np.linspace(0., 1., len(third_axis))]

znemag = np.full((len(hcojcsu),len(third_axis)),-2.)
znenex = np.full((len(hcojcsu),len(third_axis)),-2.)
hcojcind = 12
hcojc = hcojcsu[hcojcind]

ax_bl.set_xlabel(r"$\lambda$", fontsize=fs)
ax_bl.tick_params(axis='both', labelsize=fs)
ax_bl.set_ylabel(r"$\langle \sigma^z \rangle$", fontsize=fs)

ax_br.set_xlabel(r"$\lambda$", fontsize=fs)
ax_br.tick_params(axis='both', labelsize=fs)
ax_br.set_ylabel(r"$\langle n_\mathrm{ex}\rangle$", fontsize=fs)

# --- Inset inside Bottom-Left ---
axins = ax_bl.inset_axes([0.45, 0.55, 0.5, 0.4]) 
axins.plot(en_range, [en_to_lambda(e,gc=3.04438) for e in en_range], color='black')
axins.set_xlabel('$E$', fontsize=fs-2)
axins.set_ylabel(r'$\lambda$', fontsize=fs-2)
axins.tick_params(axis='both', labelsize=fs-2)

# Plotting Loop
for tqi, tq in enumerate(third_axis):
    try:
        x = np.array(datamag[(tq,hcojc)])[:,0]
        y = np.array(datamag[(tq,hcojc)])[:,1]
        params = curve_fit(fit_func, x, y)
        [C0, C2] = params[0]
        ax_bl.scatter(x, y, color=colors2[tqi], s=s)
        x_fit = np.linspace(0.,x[0],100)
        y_fit = [min([fit_func(xi, C0, C2),1.]) for xi in x_fit]
        ax_bl.plot(x_fit, y_fit, color=colors2[tqi], lw=lw)
        znemag[hcojcind,tqi] = min([fit_func(0., C0, C2),1.])

        x2 = np.array(datanex[(tq,hcojc)])[:,0]
        y2 = np.array(datanex[(tq,hcojc)])[:,1]
        params2 = curve_fit(fit_func, x2, y2)
        [C0_2, C2_2] = params2[0]
        ax_br.scatter(x2, y2, color=colors2[tqi], s=s)
        x2_fit = np.linspace(0.,x2[0],100)
        y2_fit = [max([fit_func(xi, C0_2, C2_2),0.]) for xi in x2_fit]
        ax_br.plot(x2_fit, y2_fit, color=colors2[tqi], lw=lw)
        znenex[hcojcind,tqi] = fit_func(0., C0_2, C2_2)
    except:
        znemag[hcojcind,tqi] = np.nan
        znenex[hcojcind,tqi] = np.nan

# Bottom Colorbar for Code 2
normalize2 = mcolors.Normalize(vmin=min(abs(third_axis)), vmax=max(abs(third_axis)))
scalarmappaple = cm.ScalarMappable(norm=normalize2, cmap=cmap2)
scalarmappaple.set_array(third_axis)

# Adjust 'bottom' to make room for this specific colorbar in the taller layout
cbar_ax = fig.add_axes([0.15, 0.025, 0.7, 0.01]) 
cbar2 = fig.colorbar(scalarmappaple, cax=cbar_ax, orientation='horizontal')
cbar2.set_label(third_axis_name, fontsize=fs, labelpad=10)
cbar2.ax.tick_params(labelsize=fs)

# ==========================================
# SUBPLOT LABELS (a), (b), (c)
# ==========================================
label_font = {'fontsize': fs + 4, 'fontweight': 'bold', 'va': 'bottom', 'ha': 'right'}

# (a) Top Left
ax_tl.text(-0.1, 1.02, '(a)', transform=ax_tl.transAxes, **label_font)

# (b) Top Right
# Because the image has no axes, we might need to tuck it slightly inside or adjust x
ax_tr.text(-0.05, 1.02, '(b)', transform=ax_tr.transAxes, **label_font)

# (c) Bottom
# Assuming you want the label on the bottom-left plot to represent the bottom row
ax_bl.text(-0.1, 1.02, '(c)', transform=ax_bl.transAxes, **label_font)

# Optional: If you decide you want (d) on the bottom right:
ax_br.text(-0.1, 1.02, '(d)', transform=ax_br.transAxes, **label_font)

# ==========================================
# 6. FINAL OUTPUT
# ==========================================
# rect controls [left, bottom, right, top] boundaries for the tight layout.
# Leaves bottom 10% empty so the horizontal colorbar doesn't get squished.
fig.tight_layout(rect=[0, 0.1, 1, 1])#, pad=0.2) 
fig.savefig('Fig_1.pdf', bbox_inches='tight', pad_inches=0.05)
plt.show()

/tmp/ipykernel_35522/581392382.py:122: OptimizeWarning: Covariance of the parameters could not be estimated
  params = curve_fit(fit_func, x, y)
/tmp/ipykernel_35522/581392382.py:132: OptimizeWarning: Covariance of the parameters could not be estimated
  params2 = curve_fit(fit_func, x2, y2)
/tmp/ipykernel_35522/581392382.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0.1, 1, 1])#, pad=0.2)
/tmp/ipykernel_35522/581392382.py:180: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
